In [1]:
#!/usr/bin/env python
# coding: utf-8
"""
CHB-MIT Seizure Detection Preprocessing — Expand from 3 → 11 Subjects
======================================================================
Problem: Only chb01, chb03, chb05 produced seizure detection HDF5 files.
Root causes:
  1. Summary parser regex misses variant formats like "Seizure 2 Start Time"
  2. Channel count mismatches across EDF files within a subject cause
     np.concatenate to fail
  
This script fixes both issues and processes all available subjects.
"""

import os
import re
import numpy as np
import mne
import h5py
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ============================================================================
# STEP 0: CONFIGURATION — UPDATE PATHS IF YOURS DIFFER
# ============================================================================

class CHBMITConfig:
    """Mirrors existing config. Adjust paths if needed."""
    DATA_DIR = Path("./data")
    RAW_DIR = DATA_DIR / "raw" / "chb-mit"
    PROCESSED_DIR = DATA_DIR / "processed"
    
    SFREQ_ORIGINAL = 256
    SFREQ_TARGET = 100
    L_FREQ = 0.5
    H_FREQ = 35.0
    NOTCH_FREQ = 60.0
    EPOCH_DURATION = 30.0
    LABEL_NORMAL = 0
    LABEL_SEIZURE = 1
    ARTIFACT_THRESHOLD_UV = 500
    SEED = 42
    
    # 18 standard bipolar channels shared across most CHB-MIT subjects
    STANDARD_CHANNELS = [
        'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
        'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
        'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
        'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
        'FZ-CZ', 'CZ-PZ',
    ]

CHBMITConfig.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# ============================================================================
# STEP 1: DIAGNOSTIC — WHY DID SUBJECTS FAIL?
# ============================================================================

def diagnose_all_subjects(raw_dir, processed_dir):
    """
    Check each subject: are seizure annotations parseable? Is HDF5 present?
    Prints a diagnostic table so you know exactly what needs fixing.
    """
    print("=" * 70)
    print("DIAGNOSTIC: CHB-MIT Subject Status")
    print("=" * 70)
    print(f"{'Subject':<10} {'EDF Files':<12} {'Seizures Found':<16} "
          f"{'HDF5 Exists':<14} {'Status'}")
    print("-" * 70)
    
    subjects_needing_processing = []
    
    for subj_dir in sorted(raw_dir.iterdir()):
        if not subj_dir.is_dir() or not subj_dir.name.startswith("chb"):
            continue
        
        subj = subj_dir.name
        n_edf = len(list(subj_dir.glob("*.edf")))
        
        # Parse seizures with the FIXED parser
        seizure_files = parse_summary_robust(subj_dir)
        total_seizures = sum(len(f['seizures']) for f in seizure_files)
        
        # Check if HDF5 already exists
        hdf5_exists = (processed_dir / f"chbmit_{subj}_seizure_detection.h5").exists()
        
        if hdf5_exists:
            status = "Done"
        elif total_seizures == 0:
            status = "No seizures parsed"
        else:
            status = "Needs processing"
            subjects_needing_processing.append(subj)
        
        print(f"{subj:<10} {n_edf:<12} {total_seizures:<16} "
              f"{'Yes' if hdf5_exists else 'No':<14} {status}")
    
    print("-" * 70)
    print(f"Subjects needing processing: {len(subjects_needing_processing)}")
    print(f"  → {subjects_needing_processing}")
    
    return subjects_needing_processing

In [4]:
# ============================================================================
# STEP 2: ROBUST SUMMARY PARSER — HANDLES ALL FORMAT VARIATIONS
# ============================================================================

def parse_summary_robust(subject_dir):
    """
    Parse CHB-MIT summary file with robust regex handling.
    
    CHB-MIT summary files have inconsistent formatting across subjects:
      - "Seizure Start Time: 2670 seconds"
      - "Seizure 1 Start Time: 2670 seconds"  (numbered seizures)
      - "Seizure1 Start Time: 2670 seconds"   (no space before number)
      - Extra whitespace, tabs, etc.
    
    This parser handles ALL known variations.
    
    Parameters
    ----------
    subject_dir : Path
        Path to subject folder (e.g., data/raw/chb-mit/chb01/)
    
    Returns
    -------
    list of dict
        Each dict has 'filename' (str) and 'seizures' (list of 
        {'start': int, 'end': int} in seconds from file start).
    """
    subject_dir = Path(subject_dir)
    subj_id = subject_dir.name
    summary_file = subject_dir / f"{subj_id}-summary.txt"
    
    if not summary_file.exists():
        print(f"No summary file for {subj_id}")
        return []
    
    with open(summary_file, 'r') as f:
        content = f.read()
    
    # Split into per-file blocks.
    # Each block starts with "File Name: chbXX_YY.edf" or similar
    file_blocks = re.split(r'(?=File Name:)', content)
    
    files_info = []
    
    for block in file_blocks:
        block = block.strip()
        if not block:
            continue
        
        # Extract filename
        fname_match = re.search(r'File Name:\s*(\S+\.edf)', block, re.IGNORECASE)
        if not fname_match:
            continue
        
        filename = fname_match.group(1)
        
        # Extract number of seizures
        # Handles: "Number of Seizures in File: 1"
        n_sz_match = re.search(r'Number of Seizures in File:\s*(\d+)', block)
        n_seizures = int(n_sz_match.group(1)) if n_sz_match else 0
        
        seizures = []
        
        if n_seizures > 0:
            # Extract ALL seizure start times
            # Handles: "Seizure Start Time: 100 seconds"
            #          "Seizure 1 Start Time: 100 seconds"
            #          "Seizure2 Start Time: 100 seconds"
            starts = re.findall(
                r'Seizure\s*\d*\s*Start\s*Time:\s*(\d+)\s*seconds',
                block, re.IGNORECASE
            )
            
            # Extract ALL seizure end times
            ends = re.findall(
                r'Seizure\s*\d*\s*End\s*Time:\s*(\d+)\s*seconds',
                block, re.IGNORECASE
            )
            
            # Pair them up
            for i in range(min(len(starts), len(ends))):
                s, e = int(starts[i]), int(ends[i])
                if e > s:  # Sanity check
                    seizures.append({'start': s, 'end': e})
        
        files_info.append({
            'filename': filename,
            'n_seizures_declared': n_seizures,
            'seizures': seizures
        })
    
    return files_info

In [5]:
# ============================================================================
# STEP 3: EDF LOADER WITH CHANNEL STANDARDIZATION
# ============================================================================

def load_edf_standardized(edf_path, config):
    """
    Load an EDF file and standardize channels.
    
    Handles the channel naming variations in CHB-MIT:
      - Some files have 23 channels, some 28-29
      - Channel names may have spaces: "FP1-F7" vs "FP1 - F7"
      - Some files include ECG, EMG, or extra channels
    
    We keep only the 18 standard bipolar EEG channels that are
    consistent across (almost) all subjects.
    
    Parameters
    ----------
    edf_path : Path
        Full path to the .edf file
    config : CHBMITConfig
        Configuration object
    
    Returns
    -------
    mne.io.Raw or None
        Loaded and filtered Raw object, or None if loading fails.
    """
    try:
        raw = mne.io.read_raw_edf(str(edf_path), preload=True, verbose=False)
    except Exception as e:
        print(f"    ✗ Failed to load {edf_path.name}: {e}")
        return None
    
    # Normalize channel names: strip spaces, uppercase, remove dots
    # e.g., "FP1 - F7" → "FP1-F7", "EEG FP1-F7" → "FP1-F7"
    rename_map = {}
    for ch in raw.ch_names:
        clean = ch.upper().replace(' ', '').replace('.', '')
        # Strip common prefixes like "EEG" if present
        clean = re.sub(r'^EEG', '', clean).strip('-').strip()
        rename_map[ch] = clean
    
    raw.rename_channels(rename_map)
    
    # Find which standard channels are present in this file
    available_standard = [
        ch for ch in config.STANDARD_CHANNELS 
        if ch in raw.ch_names
    ]
    
    if len(available_standard) < 10:
        # Too few channels — skip this file
        print(f"    ✗ {edf_path.name}: only {len(available_standard)}/18 "
              f"standard channels found, skipping")
        return None
    
    # Pick only the standard channels
    raw.pick_channels(available_standard, ordered=True)
    
    # Apply filters
    raw.notch_filter(config.NOTCH_FREQ, verbose=False)
    raw.filter(config.L_FREQ, config.H_FREQ, verbose=False)
    
    # Resample from 256 Hz → 100 Hz
    if raw.info['sfreq'] != config.SFREQ_TARGET:
        raw.resample(config.SFREQ_TARGET, verbose=False)
    
    return raw

In [6]:
# ============================================================================
# STEP 4: EPOCH EXTRACTION FOR SEIZURE DETECTION
# ============================================================================

def extract_seizure_epochs(raw, seizures, config):
    """
    Extract labeled 30-second epochs from one EDF file.
    
    For seizure detection, we label each epoch as:
      - SEIZURE (1) if any part of the epoch overlaps with a seizure
      - NORMAL  (0) otherwise
    
    Parameters
    ----------
    raw : mne.io.Raw
        Preprocessed Raw object (already filtered, resampled, channel-standardized)
    seizures : list of dict
        Seizure annotations [{'start': sec, 'end': sec}, ...]
    config : CHBMITConfig
        Configuration object
    
    Returns
    -------
    epochs_array : np.ndarray, shape (n_epochs, n_channels, n_samples)
    labels : np.ndarray, shape (n_epochs,)
    """
    sfreq = config.SFREQ_TARGET
    epoch_samples = int(config.EPOCH_DURATION * sfreq)  # 30s * 100Hz = 3000
    n_channels = len(raw.ch_names)
    total_samples = raw.n_times
    
    # Total number of non-overlapping epochs in this file
    n_epochs = total_samples // epoch_samples
    
    if n_epochs == 0:
        return np.array([]), np.array([])
    
    # Get raw data matrix: (n_channels, n_total_samples)
    data = raw.get_data()
    
    epochs_list = []
    labels_list = []
    
    for i in range(n_epochs):
        start_sample = i * epoch_samples
        end_sample = start_sample + epoch_samples
        
        # Time boundaries of this epoch (in seconds)
        epoch_start_sec = start_sample / sfreq
        epoch_end_sec = end_sample / sfreq
        
        # Extract epoch data
        epoch_data = data[:, start_sample:end_sample]
        
        # Check dimensions
        if epoch_data.shape != (n_channels, epoch_samples):
            continue
        
        # Z-score normalize per channel
        means = epoch_data.mean(axis=1, keepdims=True)
        stds = epoch_data.std(axis=1, keepdims=True)
        stds[stds < 1e-8] = 1e-8  # Avoid division by zero
        epoch_data = (epoch_data - means) / stds
        
        # Artifact rejection: skip if any channel exceeds threshold
        # (after z-score, threshold is in standard deviations)
        if np.any(np.abs(epoch_data) > 10):  # 10 SD = extreme artifact
            continue
        
        # Label: does this epoch overlap with any seizure?
        label = config.LABEL_NORMAL
        for sz in seizures:
            # Overlap check: epoch [epoch_start, epoch_end] ∩ seizure [sz_start, sz_end]
            if epoch_start_sec < sz['end'] and epoch_end_sec > sz['start']:
                label = config.LABEL_SEIZURE
                break
        
        epochs_list.append(epoch_data)
        labels_list.append(label)
    
    if len(epochs_list) == 0:
        return np.array([]), np.array([])
    
    return np.array(epochs_list, dtype=np.float32), np.array(labels_list, dtype=np.int32)

In [7]:
# ============================================================================
# STEP 5: PROCESS ONE SUBJECT END-TO-END
# ============================================================================

def process_subject_seizure_detection(subject_id, config):
    """
    Process a single CHB-MIT subject for seizure detection.
    
    Steps:
      1. Parse summary file for seizure annotations
      2. For each EDF file that contains seizures:
         a. Load and standardize channels
         b. Filter and resample
         c. Extract labeled epochs
      3. Also include some normal (non-seizure) files for class balance
      4. Concatenate all epochs
    
    Parameters
    ----------
    subject_id : str
        e.g., "chb01"
    config : CHBMITConfig
    
    Returns
    -------
    dict with keys: 'epochs', 'labels', 'subject_id', 'n_channels', 
                    'sfreq', 'epoch_duration', 'channel_names'
    or None if processing fails.
    """
    subj_dir = config.RAW_DIR / subject_id
    
    if not subj_dir.exists():
        print(f"  ✗ {subject_id}: directory not found")
        return None
    
    print(f"\n  Processing {subject_id}...")
    
    # --- 1. Parse seizure annotations ---
    files_info = parse_summary_robust(subj_dir)
    
    if not files_info:
        print(f"  ✗ {subject_id}: no files parsed from summary")
        return None
    
    # Separate files with and without seizures
    seizure_files = [f for f in files_info if len(f['seizures']) > 0]
    normal_files = [f for f in files_info if len(f['seizures']) == 0]
    
    total_sz = sum(len(f['seizures']) for f in seizure_files)
    print(f"    Found {total_sz} seizures across {len(seizure_files)} files "
          f"(+ {len(normal_files)} normal files)")
    
    if total_sz == 0:
        print(f"{subject_id}: no seizures found — check summary file format")
        return None
    
    all_epochs = []
    all_labels = []
    channel_names = None
    
    # --- 2. Process seizure files (all of them) ---
    for finfo in seizure_files:
        edf_path = subj_dir / finfo['filename']
        if not edf_path.exists():
            print(f"    ✗ Missing: {finfo['filename']}")
            continue
        
        raw = load_edf_standardized(edf_path, config)
        if raw is None:
            continue
        
        if channel_names is None:
            channel_names = raw.ch_names
        
        epochs, labels = extract_seizure_epochs(raw, finfo['seizures'], config)
        
        if len(epochs) > 0:
            # Ensure consistent channel count
            if epochs.shape[1] == len(channel_names):
                all_epochs.append(epochs)
                all_labels.append(labels)
                n_sz = np.sum(labels == config.LABEL_SEIZURE)
                print(f"    ✓ {finfo['filename']}: {len(epochs)} epochs "
                      f"({n_sz} seizure, {len(epochs)-n_sz} normal)")
            else:
                print(f"    ✗ {finfo['filename']}: channel mismatch "
                      f"({epochs.shape[1]} vs {len(channel_names)}), skipped")
    
    # --- 3. Add some normal files for better class balance ---
    # Include up to 3 normal files (picked evenly from the list)
    n_normal_to_add = min(3, len(normal_files))
    if n_normal_to_add > 0:
        # Pick evenly spaced normal files
        indices = np.linspace(0, len(normal_files)-1, n_normal_to_add, dtype=int)
        selected_normal = [normal_files[i] for i in indices]
        
        for finfo in selected_normal:
            edf_path = subj_dir / finfo['filename']
            if not edf_path.exists():
                continue
            
            raw = load_edf_standardized(edf_path, config)
            if raw is None:
                continue
            
            epochs, labels = extract_seizure_epochs(raw, [], config)
            
            if len(epochs) > 0 and epochs.shape[1] == len(channel_names):
                all_epochs.append(epochs)
                all_labels.append(labels)
                print(f"    ✓ {finfo['filename']}: {len(epochs)} normal epochs (balance)")
    
    # --- 4. Concatenate ---
    if not all_epochs:
        print(f"  ✗ {subject_id}: no valid epochs extracted")
        return None
    
    epochs_combined = np.concatenate(all_epochs, axis=0)
    labels_combined = np.concatenate(all_labels, axis=0)
    
    n_seizure = np.sum(labels_combined == config.LABEL_SEIZURE)
    n_normal = np.sum(labels_combined == config.LABEL_NORMAL)
    
    print(f"    ═══ {subject_id} TOTAL: {len(labels_combined)} epochs "
          f"({n_seizure} seizure, {n_normal} normal) "
          f"| shape: {epochs_combined.shape}")
    
    return {
        'epochs': epochs_combined,
        'labels': labels_combined,
        'subject_id': subject_id,
        'n_channels': epochs_combined.shape[1],
        'sfreq': config.SFREQ_TARGET,
        'epoch_duration': config.EPOCH_DURATION,
        'channel_names': channel_names,
    }

In [8]:
# ============================================================================
# STEP 6: SAVE TO HDF5
# ============================================================================

def save_subject_hdf5(data, output_path):
    """
    Save one subject's seizure detection data to HDF5.
    
    Parameters
    ----------
    data : dict
        Output from process_subject_seizure_detection()
    output_path : Path
        Where to save the .h5 file
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    with h5py.File(output_path, 'w') as f:
        f.create_dataset('epochs', data=data['epochs'], compression='gzip')
        f.create_dataset('labels', data=data['labels'])
        f.create_dataset('subject_ids', 
                         data=np.array([data['subject_id']] * len(data['labels']),
                                       dtype='S20'))
        # Metadata
        f.attrs['subject_id'] = data['subject_id']
        f.attrs['n_channels'] = data['n_channels']
        f.attrs['sfreq'] = data['sfreq']
        f.attrs['epoch_duration'] = data['epoch_duration']
        f.attrs['n_epochs'] = len(data['labels'])
        f.attrs['n_seizure'] = int(np.sum(data['labels'] == 1))
        f.attrs['n_normal'] = int(np.sum(data['labels'] == 0))
        f.attrs['dataset'] = 'chb-mit'
        f.attrs['mode'] = 'seizure_detection'
        if data['channel_names']:
            f.attrs['ch_names'] = ','.join(data['channel_names'])
    
    size_mb = output_path.stat().st_size / (1024 * 1024)
    print(f"Saved: {output_path.name} ({size_mb:.1f} MB)")

In [9]:
# ============================================================================
# STEP 7: COMBINE ALL SUBJECTS INTO ONE FILE
# ============================================================================

def combine_all_seizure_detection(processed_dir, config):
    """
    Combine all per-subject seizure detection HDF5 files into one.
    Handles variable channel counts by padding/truncating to the
    minimum shared channel count.
    
    Parameters
    ----------
    processed_dir : Path
    config : CHBMITConfig
    
    Returns
    -------
    Path to the combined HDF5 file
    """
    files = sorted(processed_dir.glob("chbmit_chb*_seizure_detection.h5"))
    
    if not files:
        print("No seizure detection files found to combine.")
        return None
    
    print(f"\nCombining {len(files)} subject files...")
    
    # First pass: find minimum channel count across all subjects
    min_channels = float('inf')
    for f in files:
        with h5py.File(f, 'r') as hf:
            min_channels = min(min_channels, hf['epochs'].shape[1])
    
    print(f"  Using {min_channels} channels (minimum across subjects)")
    
    # Second pass: load and truncate to min_channels
    all_epochs = []
    all_labels = []
    all_subject_ids = []
    
    for f in files:
        with h5py.File(f, 'r') as hf:
            epochs = hf['epochs'][:, :min_channels, :]  # Truncate channels
            labels = hf['labels'][:]
            # Handle old files that don't have subject_ids dataset
            if 'subject_ids' in hf:
                subj_ids = hf['subject_ids'][:]
            else:
                # Infer subject ID from filename: chbmit_chb01_seizure_detection.h5
                subj_name = f.stem.split('_')[1]  # e.g., "chb01"
                subj_ids = np.array([subj_name.encode()] * len(labels), dtype='S20')
            
            all_epochs.append(epochs)
            all_labels.append(labels)
            all_subject_ids.append(subj_ids)
            
            n_sz = np.sum(labels == 1)
            print(f"  ✓ {f.name}: {len(labels)} epochs ({n_sz} seizure)")
    
    combined_epochs = np.concatenate(all_epochs, axis=0)
    combined_labels = np.concatenate(all_labels, axis=0)
    combined_ids = np.concatenate(all_subject_ids, axis=0)
    
    # Save combined file
    output_path = processed_dir / "chbmit_combined_seizure_detection.h5"
    
    with h5py.File(output_path, 'w') as f:
        f.create_dataset('epochs', data=combined_epochs, compression='gzip')
        f.create_dataset('labels', data=combined_labels)
        f.create_dataset('subject_ids', data=combined_ids)
        f.attrs['n_channels'] = min_channels
        f.attrs['sfreq'] = config.SFREQ_TARGET
        f.attrs['epoch_duration'] = config.EPOCH_DURATION
        f.attrs['n_epochs'] = len(combined_labels)
        f.attrs['n_seizure'] = int(np.sum(combined_labels == 1))
        f.attrs['n_normal'] = int(np.sum(combined_labels == 0))
        f.attrs['n_subjects'] = len(files)
        f.attrs['dataset'] = 'chb-mit'
        f.attrs['mode'] = 'seizure_detection'
    
    total_sz = np.sum(combined_labels == 1)
    total_norm = np.sum(combined_labels == 0)
    size_mb = output_path.stat().st_size / (1024 * 1024)
    
    print(f"\nCombined: {output_path.name} ({size_mb:.1f} MB)")
    print(f"     {len(combined_labels)} total epochs "
          f"({total_sz} seizure, {total_norm} normal)")
    print(f"     {len(files)} subjects, {min_channels} channels")
    
    return output_path

In [10]:
# ============================================================================
# STEP 8: MAIN EXECUTION
# ============================================================================

if __name__ == "__main__" or True:  # Always run in notebook
    
    config = CHBMITConfig
    
    # --- Phase 1: Diagnose ---
    print("\n" + "=" * 70)
    print("PHASE 1: DIAGNOSTIC")
    print("=" * 70)
    subjects_to_process = diagnose_all_subjects(config.RAW_DIR, config.PROCESSED_DIR)
    
    # --- Phase 2: Process missing subjects ---
    print("\n" + "=" * 70)
    print("PHASE 2: PROCESSING MISSING SUBJECTS")
    print("=" * 70)
    
    if not subjects_to_process:
        print("All subjects already processed! Nothing to do.")
    else:
        successful = 0
        failed = 0
        
        for subj in subjects_to_process:
            result = process_subject_seizure_detection(subj, config)
            
            if result is not None:
                # Save individual subject HDF5
                out_path = config.PROCESSED_DIR / f"chbmit_{subj}_seizure_detection.h5"
                save_subject_hdf5(result, out_path)
                successful += 1
            else:
                failed += 1
        
        print(f"\n{'=' * 70}")
        print(f"PHASE 2 RESULTS: {successful} succeeded, {failed} failed")
        print(f"{'=' * 70}")
    
    # --- Phase 3: Rebuild combined file ---
    print("\n" + "=" * 70)
    print("PHASE 3: REBUILDING COMBINED DATASET")
    print("=" * 70)
    combine_all_seizure_detection(config.PROCESSED_DIR, config)
    
    # --- Phase 4: Final verification ---
    print("\n" + "=" * 70)
    print("PHASE 4: FINAL VERIFICATION")
    print("=" * 70)
    
    combined_path = config.PROCESSED_DIR / "chbmit_combined_seizure_detection.h5"
    if combined_path.exists():
        with h5py.File(combined_path, 'r') as f:
            print(f"  Epochs shape:  {f['epochs'].shape}")
            print(f"  Labels shape:  {f['labels'].shape}")
            print(f"  Subjects:      {f.attrs['n_subjects']}")
            print(f"  Seizure:       {f.attrs['n_seizure']}")
            print(f"  Normal:        {f.attrs['n_normal']}")
            print(f"  Channels:      {f.attrs['n_channels']}")
            print(f"  Sfreq:         {f.attrs['sfreq']} Hz")
            
            # Per-subject breakdown
            subj_ids = f['subject_ids'][:]
            labels = f['labels'][:]
            unique_subjs = np.unique(subj_ids)
            
            print(f"\n  Per-subject breakdown:")
            print(f"  {'Subject':<12} {'Total':<8} {'Seizure':<10} {'Normal':<10}")
            print(f"  {'-'*40}")
            for s in unique_subjs:
                mask = subj_ids == s
                n_tot = np.sum(mask)
                n_sz = np.sum(labels[mask] == 1)
                print(f"  {s.decode():<12} {n_tot:<8} {n_sz:<10} {n_tot-n_sz:<10}")
    
    print("\nCHB-MIT seizure detection expansion complete!")


PHASE 1: DIAGNOSTIC
DIAGNOSTIC: CHB-MIT Subject Status
Subject    EDF Files    Seizures Found   HDF5 Exists    Status
----------------------------------------------------------------------
chb01      42           7                Yes            Done
chb03      38           7                Yes            Done
chb05      39           5                Yes            Done
chb08      20           5                Yes            Done
chb10      25           7                Yes            Done
chb12      24           40               Yes            Done
chb14      26           8                Yes            Done
chb15      40           20               Yes            Done
chb17      21           3                Yes            Done
chb19      30           3                Yes            Done
chb20      29           8                Yes            Done
chb22      31           3                Yes            Done
----------------------------------------------------------------------
Subject

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=30d601c2-0a51-44b0-ac6c-72cf48d1679e' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>